# 02_visualizacion_recortes

## 1. Configuration

In [ ]:
from pathlib import Path
import re
import shutil
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEP = "=" * 40

# ---- Folders ----
DIR_BASE = next((c for c in (Path.cwd(), Path.cwd().parent) if (c / "metadatos").exists()), Path(".."))
DIR_META = DIR_BASE / "metadatos"
DIR_H5   = DIR_BASE / "nisar_gcov_h5"
DIR_REC  = DIR_BASE / "recortes"
DIR_FIG  = DIR_BASE / "figuras"
DIR_SUC  = DIR_BASE / "sucesos"
DIR_REC.mkdir(exist_ok=True)
DIR_FIG.mkdir(exist_ok=True)

# ---- Area: middle Atrato, UTM 18N (EPSG:32618) ----
XMIN, XMAX = 272_900, 339_900
YMIN, YMAX = 613_600, 796_400
EPSG     = 32618
RES      = 20
RES_FIG  = 60
RECORTAR_VISTA = True

# ---- Paths inside the GCOV ----
G_A   = "/science/LSAR/GCOV/grids/frequencyA"
IDENT = "/science/LSAR/identification"

# ---- Fixed scales ----
ESC = {"HH_lin": (0, 0.5), "HV_lin": (0, 0.1),
       "HH_dB": (-25, 0),  "HV_dB": (-30, -8),  "Q": (0, 15)}
PISO = 1e-3

# ---- Reference towns ----
PUEBLOS = {"Quibdó": (316.0, 629.7), "Vigía del Fuerte": (290.4, 728.7), "Murindó": (299.2, 771.8)}

# ---- Zoom at working resolution ----
ZOOM_CENTRO     = PUEBLOS["Quibdó"]
ZOOM_MEDIO_LADO = 6

LIBRE_GB = shutil.disk_usage(DIR_BASE).free / 1e9
GUARDAR  = LIBRE_GB > 2

print(SEP)
print(f"{'Archivos':<12}: {DIR_H5}")
print(f"{'Zona':<12}: x {XMIN}–{XMAX} m   y {YMIN}–{YMAX} m   (EPSG:{EPSG})")
print(f"{'Malla':<12}: {RES} m (recortes)   {RES_FIG} m (figuras)")
print(f"{'Disco libre':<12}: {LIBRE_GB:.1f} GB")
if not GUARDAR:
    print("Ojo: el disco está casi lleno. No se guardarán recortes y las figuras pueden fallar.")
print(SEP)

## 2. Inventory: what is there, its date and whether it is complete

In [ ]:
PATRON = re.compile(r"NISAR_L2_(PR|UR|OD)_GCOV_(\d{3})_(\d{3})_([AD])_(\d{3})_(\d{4})_([A-Z]{4})_[A-Z]_(\d{8}T\d{6})")
TIPOS  = {"PR": "Producción", "UR": "Urgente", "OD": "A pedido"}


def leer_texto(grupo, clave):
    if clave not in grupo:
        return ""
    v = grupo[clave][()]
    v = v.item() if hasattr(v, "item") and getattr(v, "size", 1) == 1 else v
    return (v.decode() if isinstance(v, bytes) else str(v)).strip()


def papeles_por_archivo():
    ruta = DIR_META / "fases_eventos.csv"
    if not ruta.exists():
        return {}
    papel = {}
    fases = pd.read_csv(ruta).fillna("")
    for r in fases.to_dict("records"):
        for nombre in ("pre", "durante", "post"):
            valor = r.get(f"archivos_{nombre}", r.get(f"archivo_{nombre}", ""))
            for a in str(valor).split(";"):
                a = a.strip()
                if a:
                    papel.setdefault(a, set()).add(f"{nombre} {r['evento']}")
    return {a: ", ".join(sorted(v)) for a, v in papel.items()}


def revisar(ruta, papeles):
    m = PATRON.search(ruta.name)
    if not m:
        return None
    tipo, ciclo, track, direc, frame, modo, pols, t = m.groups()
    fecha = pd.to_datetime(t, format="%Y%m%dT%H%M%S")
    fila = {"archivo": ruta.name, "fecha": fecha.date(),
            "hora_local": (fecha - pd.Timedelta(hours=5)).strftime("%H:%M"),
            "orbita": "Ascendente" if direc == "A" else "Descendente",
            "track": int(track), "frame": int(frame), "modo": modo, "pols": pols, "tipo": TIPOS[tipo],
            "stack": f"{direc}_T{int(track):03d}", "pasada": f"{direc}_T{int(track):03d}_{fecha:%Y%m%d}",
            "papel": papeles.get(ruta.name, ""), "gb": round(ruta.stat().st_size / 1e9, 2),
            "completo": False, "avance": "", "pols_h5": "", "pixel_m": np.nan, "epsg": 0, "orbita_h5": "", "inicio_utc": ""}
    try:
        with h5py.File(ruta, "r") as f:
            g = f[G_A]
            fila["completo"] = "HHHH" in g
            fila["pols_h5"]  = ",".join(p.decode() for p in g["listOfPolarizations"][()])
            fila["pixel_m"]  = float(g["xCoordinateSpacing"][()])
            fila["epsg"]     = int(g["projection"].attrs["epsg_code"])
            fila["orbita_h5"]  = leer_texto(f[IDENT], "orbitPassDirection")
            fila["inicio_utc"] = leer_texto(f[IDENT], "zeroDopplerStartTime")[:19]
            fila["avance"] = "100 %"
    except (OSError, KeyError) as e:
        total = re.search(r"stored_eof = (\d+)", str(e))
        fila["avance"] = f"{100 * ruta.stat().st_size / int(total.group(1)):.0f} %" if total else "no abre"
    return fila


papeles = papeles_por_archivo()
filas = [revisar(p, papeles) for p in sorted(DIR_H5.glob("*.h5"))]
inv = pd.DataFrame([f for f in filas if f]).sort_values(["stack", "fecha", "frame"]).reset_index(drop=True)
inv.to_csv(DIR_META / "inventario_local.csv", index=False)

for stack, grupo in inv.groupby("stack", sort=False):
    print(SEP)
    print(f"Stack {stack}  ·  órbita {grupo['orbita'].iloc[0].upper()}")
    print(f"{'Fecha':<12}{'Hora':<7}{'Fr':<4}{'Modo':<6}{'Pols':<6}{'Tipo':<12}{'GB':>6}  {'Estado':<16}Papel")
    for r in grupo.itertuples():
        estado = "completo" if r.completo else f"a medias {r.avance}"
        if r.completo and r.epsg != EPSG:
            estado = f"EPSG {r.epsg}"
        print(f"{str(r.fecha):<12}{r.hora_local:<7}{r.frame:<4}{r.modo:<6}{r.pols:<6}{r.tipo:<12}{r.gb:>6.2f}  {estado:<16}{r.papel}")
print(SEP)
print(f"{'Completos':<12}: {inv['completo'].sum()} de {len(inv)}")
print(f"{'Guardado':<12}: inventario_local.csv")

## 3. Extract the Atrato area at 20 m

In [ ]:
X0 = np.floor(XMIN / RES_FIG) * RES_FIG
X1 = np.ceil(XMAX / RES_FIG) * RES_FIG
Y0 = np.floor(YMIN / RES_FIG) * RES_FIG
Y1 = np.ceil(YMAX / RES_FIG) * RES_FIG
NX, NY = int(round((X1 - X0) / RES)), int(round((Y1 - Y0) / RES))
EXTENT_KM = [X0 / 1e3, X1 / 1e3, Y0 / 1e3, Y1 / 1e3]


def media_bloques(a, f):
    if f == 1:
        return a
    ny, nx = (a.shape[0] // f) * f, (a.shape[1] // f) * f
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(a[:ny, :nx].reshape(ny // f, f, nx // f, f), axis=(1, 3))


def leer_en_malla(ruta, banda, filas_bloque=64):
    salida = np.full((NY, NX), np.nan, dtype=np.float32)
    with h5py.File(ruta, "r") as f:
        g = f[G_A]
        x, y = g["xCoordinates"][:], g["yCoordinates"][:]
        dx, dy = x[1] - x[0], y[0] - y[1]
        F = int(round(RES / dx))
        c0 = int(round((X0 - (x[0] - dx / 2)) / dx))
        r0 = int(round(((y[0] + dy / 2) - Y1) / dy))
        ox = (-c0 + F - 1) // F if c0 < 0 else 0
        oy = (-r0 + F - 1) // F if r0 < 0 else 0
        c0, r0 = c0 + ox * F, r0 + oy * F
        nbx = min(NX - ox, (len(x) - c0) // F)
        nby = min(NY - oy, (len(y) - r0) // F)
        if nbx <= 0 or nby <= 0:
            return salida
        for i in range(0, nby, filas_bloque):
            n = min(filas_bloque, nby - i)
            rs = slice(r0 + i * F, r0 + (i + n) * F)
            cs = slice(c0, c0 + nbx * F)
            datos = g[banda][rs, cs].astype(np.float32)
            mascara = g["mask"][rs, cs]
            datos[(mascara == 0) | (mascara == 255) | ~np.isfinite(datos)] = np.nan
            salida[oy + i:oy + i + n, ox:ox + nbx] = media_bloques(datos, F)
    return salida


def extraer_pasada(pasada, filas):
    frames = "+".join(f"F{fr:03d}" for fr in sorted(filas["frame"]))
    stack, fecha = pasada[:6], f"{pasada[7:11]}-{pasada[11:13]}-{pasada[13:15]}"
    destino = DIR_REC / stack / fecha / f"{pasada}_{frames}.npz"
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        try:
            d = np.load(destino)
            return {"HH": d["HH"], "HV": d["HV"] if "HV" in d.files else None}
        except Exception:
            print(f"Recorte dañado, se vuelve a leer: {destino.name}")
    out = {}
    for banda, clave in (("HHHH", "HH"), ("HVHV", "HV")):
        m = None
        for r in filas.itertuples():
            if clave not in r.pols_h5.split(","):
                continue
            capa = leer_en_malla(DIR_H5 / r.archivo, banda)
            if m is None:
                m = capa
            else:
                hueco = np.isnan(m)
                m[hueco] = capa[hueco]
            del capa
        if m is not None:
            out[clave] = m
    if GUARDAR:
        try:
            np.savez_compressed(destino, **out)
        except OSError as e:
            print(f"No se pudo guardar {destino.name} ({e.strerror}); se sigue sin caché.")
    out.setdefault("HV", None)
    return out


def recorte_zoom(a, centro_km, medio_lado_km):
    c = int((centro_km[0] * 1e3 - X0) / RES)
    r = int((Y1 - centro_km[1] * 1e3) / RES)
    h = int(medio_lado_km * 1e3 / RES)
    r0, r1, c0, c1 = max(r - h, 0), min(r + h, NY), max(c - h, 0), min(c + h, NX)
    ext = [(X0 + c0 * RES) / 1e3, (X0 + c1 * RES) / 1e3, (Y1 - r1 * RES) / 1e3, (Y1 - r0 * RES) / 1e3]
    return a[r0:r1, c0:c1].copy(), ext


def version_ligera(a, rng):
    if a is None:
        return None
    v = a[np.isfinite(a)]
    muestra = v[rng.integers(0, v.size, 200_000)] if v.size else v
    del v
    zoom, ext = recorte_zoom(a, ZOOM_CENTRO, ZOOM_MEDIO_LADO)
    return {"fig": media_bloques(a, RES_FIG // RES), "muestra": muestra, "zoom": zoom, "ext_zoom": ext}


completos = inv[inv["completo"]].copy()
modo_comun = completos.groupby("stack")["modo"].agg(lambda s: s.mode().iloc[0])
completos["orden"] = completos.apply(lambda r: r["modo"] != modo_comun[r["stack"]], axis=1)
completos = completos.sort_values(["pasada", "orden"]).drop_duplicates(["pasada", "frame"])
for r in completos[completos["epsg"] != EPSG].itertuples():
    print(f"Se omite (EPSG:{r.epsg}, otra zona UTM; habría que reproyectar): {r.archivo[:62]}")
completos = completos[completos["epsg"] == EPSG]

rng = np.random.default_rng(0)
ESCENAS = {}
print(SEP)
print(f"{'Pasada':<20}{'Fecha':<12}{'Frames':<11}{'Pols':<7}{'Cobertura':>10}{'HH mediana':>12}  Papel")
for pasada, filas in completos.groupby("pasada"):
    datos = extraer_pasada(pasada, filas)
    meta = filas.iloc[0]
    papel = ", ".join(sorted({p.strip() for t in filas["papel"] for p in t.split(",") if p.strip()}))
    hh = datos["HH"]
    cob = np.isfinite(hh).mean() * 100
    med = 10 * np.log10(np.nanmedian(hh)) if cob > 0 else np.nan
    ESCENAS[pasada] = {"HH": version_ligera(hh, rng), "HV": version_ligera(datos["HV"], rng),
                       "meta": meta, "papel": papel, "cobertura": cob,
                       "frames": "+".join(f"F{fr:03d}" for fr in sorted(filas["frame"]))}
    del datos, hh
    pols = "HH+HV" if ESCENAS[pasada]["HV"] is not None else "HH"
    print(f"{pasada:<20}{str(meta['fecha']):<12}{ESCENAS[pasada]['frames']:<11}{pols:<7}{cob:>9.0f}%{med:>10.1f} dB  {papel}")
print(SEP)
print(f"{'Malla':<12}: {NX} x {NY} píxeles de {RES} m")
print(f"{'Recortes':<12}: {DIR_REC}")

## 4. Figure 1 – As delivered by the sensor

In [ ]:
def a_db(x):
    return 10 * np.log10(np.clip(x, PISO, None))


def guardar_fig(fig, nombre, dpi=130, sub=""):
    ruta = DIR_FIG / sub / nombre
    ruta.parent.mkdir(parents=True, exist_ok=True)
    try:
        fig.savefig(ruta, dpi=dpi, bbox_inches="tight")
    except OSError as e:
        print(f"No se pudo guardar {nombre} ({e.strerror})")
    plt.show()
    return ruta


def titulo(esc, partir=False):
    m = esc["meta"]
    papel = f" · {esc['papel']}" if esc["papel"] else ""
    corte = "\n" if partir else " · "
    return f"{m['stack']} · {m['fecha']} {m['hora_local']} (hora local){corte}{m['orbita']} · {esc['frames']}{papel}"


NOTA_VACIO = "Rayado = sin dato: esa parte de la zona quedó fuera de la franja que barrió el satélite en esa pasada."


def nota_vacio(fig):
    fig.text(0.5, -0.005, NOTA_VACIO, ha="center", va="top", fontsize=8, color="#555555")


def caja_datos(a, margen_km=2.0):
    filas = np.where(np.isfinite(a).any(axis=1))[0]
    cols = np.where(np.isfinite(a).any(axis=0))[0]
    if filas.size == 0:
        return None
    paso = RES_FIG / 1e3
    x0, x1 = EXTENT_KM[0] + cols[0] * paso, EXTENT_KM[0] + (cols[-1] + 1) * paso
    y1, y0 = EXTENT_KM[3] - filas[0] * paso, EXTENT_KM[3] - (filas[-1] + 1) * paso
    return [max(x0 - margen_km, EXTENT_KM[0]), min(x1 + margen_km, EXTENT_KM[1]),
            max(y0 - margen_km, EXTENT_KM[2]), min(y1 + margen_km, EXTENT_KM[3])]


VISTA = {}
for esc in ESCENAS.values():
    st, c = esc["meta"]["stack"], caja_datos(esc["HH"]["fig"])
    if c is not None:
        v = VISTA.get(st, c)
        VISTA[st] = [min(v[0], c[0]), max(v[1], c[1]), min(v[2], c[2]), max(v[3], c[3])]


def vista(stack):
    return VISTA.get(stack, EXTENT_KM) if RECORTAR_VISTA else EXTENT_KM


def alto_panel(ext, ancho_in):
    return ancho_in * (ext[3] - ext[2]) / (ext[1] - ext[0])


def decorar(ax, titulo_panel, extent=None):
    extent = extent or EXTENT_KM
    ax.set_facecolor("#f0f0f0")
    ax.patch.set_hatch("///")
    ax.patch.set_edgecolor("#c4c4c4")
    ax.set_title(titulo_panel, fontsize=10)
    ax.set_xlabel("Este (km)")
    ax.set_ylabel("Norte (km)")
    for nombre, (xk, yk) in PUEBLOS.items():
        if extent[0] <= xk <= extent[1] and extent[2] <= yk <= extent[3]:
            ax.plot(xk, yk, marker="o", ms=4, mfc="none", mec="yellow")
            ax.annotate(nombre, (xk, yk), xytext=(4, 4), textcoords="offset points", color="yellow", fontsize=8)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])


def fig_sensor(pasada, esc):
    bandas = [("HH", esc["HH"])] + ([("HV", esc["HV"])] if esc["HV"] is not None else [])
    n = 2 * len(bandas)
    ext = vista(esc["meta"]["stack"])
    alto = float(np.clip(alto_panel(ext, 2.6) + 1.2, 6.5, 11))
    fig = plt.figure(figsize=(3.3 * n + 8.5, alto), layout="constrained")
    gs = fig.add_gridspec(2, n + 2, width_ratios=[1] * n + [1.5, 1.6])
    for j, (nom, b) in enumerate(bandas):
        for k, (datos, clave, etiqueta) in enumerate(((b["fig"], f"{nom}_lin", "γ0 lineal"),
                                                     (a_db(b["fig"]), f"{nom}_dB", "γ0 (dB)"))):
            ax = fig.add_subplot(gs[:, j + k * len(bandas)])
            vmin, vmax = ESC[clave]
            im = ax.imshow(datos, cmap="gray", vmin=vmin, vmax=vmax, extent=EXTENT_KM)
            decorar(ax, f"{nom} · {etiqueta}", extent=ext)
            fig.colorbar(im, ax=ax, fraction=0.05, pad=0.02, label=etiqueta)
    for fila, (nom, b) in enumerate(bandas):
        ax = fig.add_subplot(gs[fila, n])
        ax.imshow(a_db(b["zoom"]), cmap="gray", vmin=ESC[f"{nom}_dB"][0], vmax=ESC[f"{nom}_dB"][1],
                  extent=b["ext_zoom"])
        decorar(ax, f"Zoom Quibdó · {nom} dB · {RES} m", extent=b["ext_zoom"])
        if np.isfinite(b["zoom"]).mean() < 0.05:
            ax.text(0.5, 0.5, "Quibdó queda fuera\nde esta pasada", transform=ax.transAxes, ha="center",
                    va="center", fontsize=9, bbox=dict(facecolor="white", edgecolor="none"))
    ax_lin, ax_db = fig.add_subplot(gs[0, n + 1]), fig.add_subplot(gs[1, n + 1])
    for nom, b in bandas:
        v = b["muestra"]
        ax_lin.hist(v, bins=200, range=(0, 0.3), histtype="step", density=True, label=nom)
        ax_db.hist(a_db(v), bins=200, range=(-35, 5), histtype="step", density=True, label=nom)
    ax_lin.set(title="Histograma lineal", xlabel="γ0 lineal", ylabel="densidad")
    ax_db.set(title="Histograma en dB", xlabel="γ0 (dB)", ylabel="densidad")
    ax_lin.legend(); ax_db.legend()
    fig.suptitle(f"Como la entrega el sensor · {titulo(esc)}", fontsize=12)
    nota_vacio(fig)
    return guardar_fig(fig, f"{pasada}_sensor.png", sub=f"por_region/{esc['meta']['stack']}/{esc['meta']['fecha']}")


figuras = []
for pasada, esc in ESCENAS.items():
    figuras.append(fig_sensor(pasada, esc))

## 5. Figure 2 – Polarimetric RGB

In [ ]:
def normalizar(a, rango):
    lo, hi = rango
    return np.clip((a - lo) / (hi - lo), 0, 1)


def rgb_polarimetrico(hh, hv):
    hh_db = a_db(hh)
    if hv is None:
        g = normalizar(hh_db, ESC["HH_dB"])
        canales = [g, g, g]
    else:
        hv_db = a_db(hv)
        canales = [normalizar(hh_db, ESC["HH_dB"]), normalizar(hv_db, ESC["HV_dB"]),
                   normalizar(hh_db - hv_db, ESC["Q"])]
    alfa = np.isfinite(hh) & (np.isfinite(hv) if hv is not None else True)
    return np.dstack([np.nan_to_num(c) for c in canales] + [alfa.astype(float)])


LEYENDA_C1 = ("R = HH   G = HV   B = HH−HV\n"
              "azul oscuro/negro: agua\namarillo verdoso: bosque\n"
              "gris violáceo: pasto o suelo\ncrema/blanco: bosque inundado o ciudad")


def fig_rgb(pasada, esc):
    hh = esc["HH"]["fig"]
    hv = esc["HV"]["fig"] if esc["HV"] is not None else None
    ext = vista(esc["meta"]["stack"])
    fig, ax = plt.subplots(figsize=(6.5, float(np.clip(alto_panel(ext, 5.0) + 1.4, 6, 12))), layout="constrained")
    ax.imshow(rgb_polarimetrico(hh, hv), extent=EXTENT_KM)
    decorar(ax, "RGB polarimétrico" if hv is not None else "Solo HH (gris)", extent=ext)
    if hv is not None:
        ax.text(0.02, 0.02, LEYENDA_C1, transform=ax.transAxes, fontsize=8, va="bottom",
                bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"))
    fig.suptitle(titulo(esc, partir=True), fontsize=11)
    nota_vacio(fig)
    return guardar_fig(fig, f"{pasada}_rgb.png", sub=f"por_region/{esc['meta']['stack']}/{esc['meta']['fecha']}")


for pasada, esc in ESCENAS.items():
    figuras.append(fig_rgb(pasada, esc))

## 6. Coverage, same-orbit dates and change RGB

In [ ]:
def fig_cobertura(por_stack):
    stacks = sorted(por_stack)
    fig, ejes = plt.subplots(1, len(stacks), figsize=(3.2 * len(stacks) + 1.5, 9.5), squeeze=False,
                             layout="constrained")
    print(SEP)
    print(f"{'Stack':<8}{'Órbita':<13}{'Fechas':>7}{'Cubre (≥1 fecha)':>18}{'Cubre (todas)':>15}")
    for ax, st in zip(ejes[0], stacks):
        conteo = sum(np.isfinite(ESCENAS[p]["HH"]["fig"]).astype(int) for p in por_stack[st])
        n = len(por_stack[st])
        alguna, todas = (conteo > 0).mean() * 100, (conteo == n).mean() * 100
        orbita = ESCENAS[por_stack[st][0]]["meta"]["orbita"]
        im = ax.imshow(np.where(conteo > 0, conteo, np.nan), cmap="viridis", vmin=1, vmax=max(n, 2),
                       extent=EXTENT_KM)
        decorar(ax, f"{st} · {orbita}\n{n} fechas · cubre {alguna:.0f} %")
        fig.colorbar(im, ax=ax, fraction=0.05, pad=0.02, label="fechas con dato")
        print(f"{st:<8}{orbita:<13}{n:>7}{alguna:>17.0f}%{todas:>14.0f}%")
    print(SEP)
    fig.suptitle("Cobertura de la zona del Atrato por stack", fontsize=12)
    nota_vacio(fig)
    return guardar_fig(fig, "cobertura_stacks.png", dpi=110, sub="general")


def fig_fechas(stack, pasadas):
    n = len(pasadas)
    ext = vista(stack)
    alto = float(np.clip(alto_panel(ext, 3.0) + 1.6, 6, 11))
    fig, ejes = plt.subplots(1, n, figsize=(3.0 * n + 1.5, alto), squeeze=False, layout="constrained")
    for ax, p in zip(ejes[0], pasadas):
        esc = ESCENAS[p]
        im = ax.imshow(a_db(esc["HH"]["fig"]), cmap="gray", vmin=ESC["HH_dB"][0], vmax=ESC["HH_dB"][1],
                       extent=EXTENT_KM)
        m = esc["meta"]
        decorar(ax, f"{m['fecha']} {m['hora_local']}\n{esc['papel'] or 'sin evento'}", extent=ext)
    fig.colorbar(im, ax=ejes[0].tolist(), fraction=0.02, pad=0.01, label="HH γ0 (dB)")
    fig.suptitle(f"Stack {stack} · HH en dB por fecha", fontsize=12)
    nota_vacio(fig)
    return guardar_fig(fig, f"{stack}_fechas_HH.png", dpi=110, sub=f"por_region/{stack}")


def fig_cambio(stack, evento, p_pre, p_dur):
    pre = a_db(ESCENAS[p_pre]["HH"]["fig"])
    dur = a_db(ESCENAS[p_dur]["HH"]["fig"])
    rgb = np.dstack([np.nan_to_num(normalizar(dur, ESC["HH_dB"])),
                     np.nan_to_num(normalizar(pre, ESC["HH_dB"])),
                     np.nan_to_num(normalizar(pre, ESC["HH_dB"])),
                     (np.isfinite(pre) & np.isfinite(dur)).astype(float)])
    ext = vista(stack)
    alto = float(np.clip(alto_panel(ext, 4.8) + 1.4, 6, 12))
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11.5, alto), layout="constrained")
    ax.imshow(rgb, extent=EXTENT_KM)
    f_pre, f_dur = ESCENAS[p_pre]["meta"]["fecha"], ESCENAS[p_dur]["meta"]["fecha"]
    decorar(ax, "RGB de cambio (C2)", extent=ext)
    ax.text(0.02, 0.02, "R = HH durante   G = B = HH pre\ncian: agua nueva\nrojo: doble rebote nuevo\ngris: sin cambio",
            transform=ax.transAxes, fontsize=8, va="bottom", bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
    im = ax2.imshow(dur - pre, cmap="RdBu_r", vmin=-10, vmax=10, extent=EXTENT_KM)
    decorar(ax2, "LR_HH = HH durante − HH pre", extent=ext)
    fig.colorbar(im, ax=ax2, fraction=0.05, pad=0.02, label="dB")
    fig.suptitle(f"Stack {stack} · Cambio {evento} · pre {f_pre} → durante {f_dur}", fontsize=12)
    nota_vacio(fig)
    return guardar_fig(fig, f"{stack}_{evento}_cambio_HH.png", sub=f"por_region/{stack}/cambios")


por_stack = {}
for p, esc in ESCENAS.items():
    por_stack.setdefault(esc["meta"]["stack"], []).append(p)

figuras.append(fig_cobertura(por_stack))

for stack, pasadas in por_stack.items():
    pasadas = sorted(pasadas)
    figuras.append(fig_fechas(stack, pasadas))
    eventos = sorted({t.split()[1] for p in pasadas for t in ESCENAS[p]["papel"].split(", ") if t})
    for ev in eventos:
        pres = [p for p in pasadas if f"pre {ev}" in ESCENAS[p]["papel"]]
        durs = [p for p in pasadas if f"durante {ev}" in ESCENAS[p]["papel"]]
        if pres and durs:
            figuras.append(fig_cambio(stack, ev, pres[-1], durs[0]))
        else:
            print(f"{stack} {ev}: falta el pre o el durante completo; no hay RGB de cambio.")

## 7. Summary

In [ ]:
print(SEP)
print(f"{'Pasadas':<12}: {len(ESCENAS)}")
for p, esc in ESCENAS.items():
    m = esc["meta"]
    print(f"  {p:<20}{str(m['fecha']):<12}{m['hora_local']:<7}{m['orbita']:<13}{esc['papel']}")
print(f"{'Figuras':<12}: {len(figuras)} en {DIR_FIG}")
for r in figuras:
    print(f"  {r.relative_to(DIR_FIG)}")
faltan = inv[~inv["completo"]]
if len(faltan):
    print(f"{'A medias':<12}: {len(faltan)} archivo(s) sin terminar de bajar")
    for r in faltan.itertuples():
        print(f"  {r.archivo[:70]:<72}{r.avance}")
print(SEP)

## 8. Organize by event

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

FASES = {"pre": "1_antes", "durante": "2_durante", "post": "3_despues"}
NOMBRE_FASE = {"pre": "antes", "durante": "durante", "post": "despues"}
PATRON_REC = re.compile(r"^([AD]_T\d{3})_(\d{8})_((?:F\d{3}\+?)+)\.npz$")
PATRON_FIG_PASADA = re.compile(r"^([AD]_T\d{3})_(\d{8})_(sensor|rgb)\.png$")
PATRON_FIG_FECHAS = re.compile(r"^([AD]_T\d{3})_fechas_HH\.png$")
PATRON_FIG_CAMBIO = re.compile(r"^([AD]_T\d{3})_(E\d)_cambio_HH\.png$")


def slug(texto):
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Za-z0-9]+", "_", t).strip("_")


def fecha_iso(yyyymmdd):
    return f"{yyyymmdd[:4]}-{yyyymmdd[4:6]}-{yyyymmdd[6:]}"


def mover(origen, destino):
    destino.parent.mkdir(parents=True, exist_ok=True)
    if origen.resolve() != destino.resolve() and not destino.exists():
        shutil.move(str(origen), str(destino))
    return destino


def enlazar(origen, destino):
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        return "ya estaba"
    try:
        os.link(origen, destino)
        return "enlace"
    except OSError:
        shutil.copy2(origen, destino)
        return "copia"


def npz_sano(ruta):
    try:
        with zipfile.ZipFile(ruta) as z:
            return len(z.namelist()) > 0
    except (zipfile.BadZipFile, OSError):
        return False


def organizar_recortes(dir_rec):
    filas = []
    for ruta in dir_rec.rglob("*.npz"):
        if any(p in ("_parciales", "_danados") for p in ruta.relative_to(dir_rec).parts):
            continue
        m = PATRON_REC.match(ruta.name)
        if not m:
            continue
        stack, fecha, frames = m.groups()
        filas.append({"ruta": ruta, "stack": stack, "fecha": fecha_iso(fecha), "frames": frames,
                      "n_frames": frames.count("F"), "bytes": ruta.stat().st_size, "sano": npz_sano(ruta)})
    rec = pd.DataFrame(filas)
    if rec.empty:
        return rec
    for r in rec[~rec["sano"]].itertuples():
        mover(r.ruta, dir_rec / "_danados" / r.ruta.name)
        print(f"{'Dañado':<12}: {r.ruta.name} → recortes/_danados/")
    rec = rec[rec["sano"]].sort_values(["stack", "fecha", "n_frames", "bytes"], ascending=[True, True, False, False])
    elegidos = rec.drop_duplicates(["stack", "fecha"])
    for r in rec.drop(elegidos.index).itertuples():
        mover(r.ruta, dir_rec / "_parciales" / r.ruta.name)
        print(f"{'Parcial':<12}: {r.ruta.name} → recortes/_parciales/ (hay uno con más frames)")
    nuevas = []
    for r in elegidos.itertuples():
        nuevas.append(mover(r.ruta, dir_rec / r.stack / r.fecha / r.ruta.name))
    elegidos = elegidos.assign(ruta=nuevas).drop(columns=["sano", "bytes"])
    return elegidos.reset_index(drop=True)


def organizar_figuras(dir_fig):
    for ruta in list(dir_fig.glob("*.png")):
        n = ruta.name
        if m := PATRON_FIG_PASADA.match(n):
            mover(ruta, dir_fig / "por_region" / m[1] / fecha_iso(m[2]) / n)
        elif m := PATRON_FIG_FECHAS.match(n):
            mover(ruta, dir_fig / "por_region" / m[1] / n)
        elif m := PATRON_FIG_CAMBIO.match(n):
            mover(ruta, dir_fig / "por_region" / m[1] / "cambios" / n)
        else:
            mover(ruta, dir_fig / "general" / n)


def horas_locales(dir_base):
    ruta = dir_base / "inventario_local.csv"
    if not ruta.exists():
        return {}
    inv = pd.read_csv(ruta).sort_values(["pasada", "hora_local"])
    return inv.drop_duplicates("pasada").set_index("pasada")["hora_local"].to_dict()


def organizar_sucesos(dir_base, recortes):
    dir_fig, dir_suc = dir_base / "figuras" / "por_region", dir_base / "sucesos"
    fases = pd.read_csv(dir_base / "fases_eventos.csv").fillna("")
    horas = horas_locales(dir_base)
    rec = {(r.stack, r.fecha): r.ruta for r in recortes.itertuples()}
    resumen, modos = [], {}
    eventos = fases.groupby("evento").filter(lambda g: (g["durante"] != "").any())

    for evento, grupo in eventos.groupby("evento"):
        f_evento = pd.Timestamp(grupo["fecha_evento"].iloc[0])
        carpeta = dir_suc / f"{evento}_{f_evento:%Y-%m-%d}_{slug(grupo['lugar'].iloc[0])}"
        filas = []
        for r in grupo.itertuples():
            for fase, sub in FASES.items():
                for fecha in [f.strip() for f in str(getattr(r, fase)).split(",") if f.strip()]:
                    pasada = f"{r.stack}_{fecha.replace('-', '')}"
                    hora = horas.get(pasada, "00:00")
                    dias = (pd.Timestamp(fecha) - f_evento).days
                    destinos = [carpeta / "por_region" / r.stack / sub / fecha,
                                carpeta / "cronologico" /
                                f"{fecha}_{hora.replace(':', 'h')}_{r.stack}_{NOMBRE_FASE[fase]}_dia{dias:+d}"]
                    archivos = []
                    origen = rec.get((r.stack, fecha))
                    figs = sorted((dir_fig / r.stack / fecha).glob(f"{pasada}_*.png"))
                    for o in ([origen] if origen else []) + figs:
                        for d in destinos:
                            k = enlazar(o, d / o.name)
                            modos[k] = modos.get(k, 0) + 1
                        archivos.append(o.name)
                    filas.append({"evento": evento, "stack": r.stack, "orbita": r.orbita, "fase": NOMBRE_FASE[fase],
                                  "fecha": fecha, "hora_local": hora, "dias_al_evento": dias,
                                  "recorte": origen.name if origen else "FALTA",
                                  "figuras": "; ".join(f.name for f in figs)})
            cambio = dir_fig / r.stack / "cambios" / f"{r.stack}_{evento}_cambio_HH.png"
            if cambio.exists():
                enlazar(cambio, carpeta / "por_region" / r.stack / "cambio" / cambio.name)
        indice = pd.DataFrame(filas).sort_values(["fecha", "hora_local", "stack"])
        carpeta.mkdir(parents=True, exist_ok=True)
        indice.to_csv(carpeta / f"indice_{evento}.csv", index=False)
        resumen.append((carpeta.name, indice))

    print(SEP)
    for nombre, indice in resumen:
        print(f"{nombre}")
        print(f"  {'Fecha':<12}{'Hora':<7}{'Días':>5}  {'Stack':<8}{'Fase':<9}Recorte")
        for r in indice.itertuples():
            print(f"  {r.fecha:<12}{r.hora_local:<7}{r.dias_al_evento:>+5d}  {r.stack:<8}{r.fase:<9}{r.recorte}")
    print(SEP)
    print(f"{'Archivos':<12}: " + ", ".join(f"{v} {k}" for k, v in modos.items()))
    return resumen


def organizar(dir_base):
    dir_base = Path(dir_base)
    recortes = organizar_recortes(dir_base / "recortes")
    organizar_figuras(dir_base / "figuras")
    return organizar_sucesos(dir_base, recortes)


SUCESOS = organizar(DIR_BASE)